# Roadmap

This document specifies the planned architecture for the new `Sampler` API, along with how it is used in `pymc.sample`, how it connects to `Monitor` and `Storage` objects.

## End goal

We aim to fulfill 5 main objectives:

1. Make `pymc.sample` arguments more robust and delegate flow to modular components
2. Have internal and external samplers wrapped under a uniform `Sampler` API
3. Define an `InitialPoint` API
4. Define a `Monitor` API that will be responsible for rendering sampling progress
5. Define a `Storage` API that will be responsible for holding samples and sampling state information
6. Define how `Monitor` and `Storage` objects interact with `Sampler`

At the end of the day, we would like to make it easier for users to write plug their own samplers into pymc and have them work.

**Optional but nice to have:**

- Async sampling
- Interruptable and resumable sampling

## What `pymc.sample` will look like after the change

In [ ]:
def sample(
    *,
    model: Model | None=None,
    sampler: Sampler | None = None,
    monitor: Monitor | None = None,
    storage: Storage | None = None,
    sampler_state: SamplerState | None = None,
    init: InitStrategy | None = None,
    blocking: bool = True,
    some_other_fixed_kwargs,
) -> tuple[Awaitable[Storage], Awaitable[SamplerState]] | tuple[Storage, SamplerState]:
    # Default values stage
    if storage is None:
        storage = ... # Build default storage
    else:
        sampler, sampler_state = load_sampler_from_storage(storage)
    if sampler is None:
        sampler = ... # Build default sampler
    if sampler_state is None:
        if init is None:
            init = ... # Build default init
        init_val = init()
        sampler_state = sampler.init(model, init_val)
    if monitor is None:
        monitor = ... # Build default monitor (sampler dependent)

    # Sampler connection stage
    sampler.connect_storage(storage)
    sampler.connect_monitor(monitor)

    # Actual work is potentially async
    future_storage, future_sampling_state = sampler.sample(**TBD)  # The signature here is to be discussed
    if blocking:
        await (future_storage, future_sampling_state)
        future_storage, future_sampling_state
    return (future_storage, future_sampling_state)